# Source database
Creates the `sales` schema in Azure SQL, enables SQL Server CDC on `sales.customers` and loads a TPC-H subset: customers, products and orders. Customers are inserted after CDC is on, so the first pipeline run gets them as change events. Clears the Delta targets.

In [ ]:
%pip install -q python-tds "pyOpenSSL<24.3" certifi

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("sql_host", "")
dbutils.widgets.text("sql_database", "")
dbutils.widgets.text("sql_user", "")
dbutils.widgets.text("secret_scope", "lab10")
dbutils.widgets.text("secret_key", "azure-sql-password")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("n_customers", "1000")

from contextlib import closing

import certifi
import pytds


def connect():
    return closing(pytds.connect(
        dsn=dbutils.widgets.get("sql_host"),
        port=1433,
        database=dbutils.widgets.get("sql_database"),
        user=dbutils.widgets.get("sql_user"),
        password=dbutils.secrets.get(dbutils.widgets.get("secret_scope"), dbutils.widgets.get("secret_key")),
        autocommit=True,
        cafile=certifi.where(),
        validate_host=True,
    ))


def insert_rows(cur, table, columns, rows, max_params=2000):
    batch = max(1, max_params // len(columns))
    row_sql = "(" + ", ".join(["%s"] * len(columns)) + ")"
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        sql = f"INSERT INTO {table} ({', '.join(columns)}) VALUES " + ", ".join([row_sql] * len(chunk))
        cur.execute(sql, tuple(v for row in chunk for v in row))

In [ ]:
DDL = [
    "IF SCHEMA_ID('sales') IS NULL EXEC('CREATE SCHEMA sales')",
    """IF EXISTS (SELECT 1 FROM sys.databases WHERE name = DB_NAME() AND is_cdc_enabled = 1)
       EXEC('IF EXISTS (SELECT 1 FROM cdc.change_tables WHERE source_object_id = OBJECT_ID(''sales.customers''))
             EXEC sys.sp_cdc_disable_table @source_schema = N''sales'', @source_name = N''customers'', @capture_instance = N''all''')""",
    "DROP TABLE IF EXISTS sales.orders",
    "DROP TABLE IF EXISTS sales.products",
    "DROP TABLE IF EXISTS sales.customers",
    """CREATE TABLE sales.customers (
        customer_id INT NOT NULL PRIMARY KEY,
        name NVARCHAR(100) NOT NULL,
        segment NVARCHAR(20) NOT NULL,
        country NVARCHAR(40) NOT NULL,
        email NVARCHAR(120) NOT NULL,
        credit_limit DECIMAL(12, 2) NOT NULL)""",
    """CREATE TABLE sales.products (
        product_id INT NOT NULL PRIMARY KEY,
        name NVARCHAR(100) NOT NULL,
        category NVARCHAR(40) NOT NULL,
        list_price DECIMAL(12, 2) NOT NULL)""",
    """CREATE TABLE sales.orders (
        order_id INT NOT NULL PRIMARY KEY,
        customer_id INT NOT NULL REFERENCES sales.customers (customer_id),
        order_date DATE NOT NULL,
        status NCHAR(1) NOT NULL,
        priority NVARCHAR(20) NOT NULL,
        total_amount DECIMAL(12, 2) NOT NULL)""",
    "IF NOT EXISTS (SELECT 1 FROM sys.databases WHERE name = DB_NAME() AND is_cdc_enabled = 1) EXEC sys.sp_cdc_enable_db",
    """EXEC sys.sp_cdc_enable_table @source_schema = N'sales', @source_name = N'customers',
       @role_name = NULL, @supports_net_changes = 1""",
]

with connect() as conn:
    cur = conn.cursor()
    for statement in DDL:
        cur.execute(statement)

In [ ]:
from pyspark.sql import functions as F

n = int(dbutils.widgets.get("n_customers"))
customers = (
    spark.table("samples.tpch.customer").where(F.col("c_custkey") <= n)
    .join(spark.table("samples.tpch.nation"), F.col("c_nationkey") == F.col("n_nationkey"))
    .select(
        F.col("c_custkey").cast("int"),
        F.col("c_name"),
        F.initcap("c_mktsegment"),
        F.initcap("n_name"),
        F.concat(F.lower(F.regexp_replace("c_name", "#", ".")), F.lit("@example.com")),
        F.greatest(F.col("c_acctbal"), F.lit(0)).cast("decimal(12,2)"),
    )
    .collect()
)
orders = (
    spark.table("samples.tpch.orders").where(F.col("o_custkey") <= n)
    .select(
        F.col("o_orderkey").cast("int"),
        F.col("o_custkey").cast("int"),
        F.add_months("o_orderdate", 32 * 12),
        F.col("o_orderstatus"),
        F.regexp_extract("o_orderpriority", r"^\d-(.*)$", 1),
        F.col("o_totalprice").cast("decimal(12,2)"),
    )
    .collect()
)
products = (
    spark.table("samples.tpch.part").where(F.col("p_partkey") <= 2000)
    .select(F.col("p_partkey").cast("int"), F.col("p_name"), F.col("p_type"), F.col("p_retailprice").cast("decimal(12,2)"))
    .collect()
)

with connect() as conn:
    cur = conn.cursor()
    insert_rows(cur, "sales.customers", ["customer_id", "name", "segment", "country", "email", "credit_limit"], customers)
    insert_rows(cur, "sales.products", ["product_id", "name", "category", "list_price"], products)
    insert_rows(cur, "sales.orders", ["order_id", "customer_id", "order_date", "status", "priority", "total_amount"], orders)

print(f"customers: {len(customers)}, products: {len(products)}, orders: {len(orders)}")

In [ ]:
import time

with connect() as conn:
    cur = conn.cursor()
    for _ in range(60):
        cur.execute("SELECT COUNT(*) FROM cdc.sales_customers_CT WHERE __$operation = 2")
        captured = cur.fetchone()[0]
        if captured >= len(customers):
            break
        time.sleep(5)
print(f"insert events captured by CDC: {captured}")
assert captured >= len(customers), "CDC capture did not catch up"

target = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
for table in ("customers_current", "customers_history", "customer_changes", "cdc_watermark", "orders_snapshot"):
    spark.sql(f"DROP TABLE IF EXISTS {target}.{table}")